[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch01/NN_DL_ch01_CreditMLP/NN_DL_ch01_CreditMLP.ipynb)

# NN_DL_ch01_CreditMLP

**MLP for Credit Scoring using the German Credit Dataset**

We load the real German Credit dataset from OpenML, preprocess it, train a multi-layer perceptron in PyTorch, and evaluate with ROC curves, confusion matrices, and feature importance analysis.

In [1]:
!pip install -q torch scikit-learn matplotlib

In [2]:
# Style & Reproducibility
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MAIN_BLUE = '#1A3A6E'
IDA_RED   = '#CD0000'
FOREST    = '#2E7D32'
CRIMSON   = '#DC3545'
AMBER     = '#FFC107'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 14,
    'figure.figsize': (10, 6),
})
np.random.seed(42)

def save_fig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight', dpi=300, transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', dpi=150, transparent=True)
    plt.show()
    print(f'Saved {name}.pdf and {name}.png')

In [3]:
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [4]:
# Load German Credit Data
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd

data = fetch_openml('credit-g', version=1, as_frame=True, parser='auto')
df = data.frame
print(f"Dataset shape: {df.shape}")
print(f"Target distribution:\n{df['class'].value_counts()}")

# Encode categoricals
le_dict = {}
X = df.drop('class', axis=1).copy()
for col in X.select_dtypes(include='category').columns:
    le_dict[col] = LabelEncoder()
    X[col] = le_dict[col].fit_transform(X[col].astype(str))

y = (df['class'] == 'bad').astype(int).values
feature_names = list(X.columns)
X = X.values.astype(np.float32)

scaler = StandardScaler()
X = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"\nTrain: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Bad credit rate: train={y_train.mean():.3f}, test={y_test.mean():.3f}")

Dataset shape: (1000, 21)
Target distribution:
class
good    700
bad     300
Name: count, dtype: int64

Train: 800, Test: 200
Bad credit rate: train=0.300, test=0.300


## Logistic Regression Baseline


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_proba = lr.predict_proba(X_test)[:, 1]
lr_pred = lr.predict(X_test)
print(f"Logistic Regression: AUC={roc_auc_score(y_test, lr_proba):.3f}, Acc={accuracy_score(y_test, lr_pred):.3f}, F1={f1_score(y_test, lr_pred):.3f}")
print("Slides: AUC~0.76, Acc~0.74")


In [5]:
# MLP Model
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class CreditMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1))
    def forward(self, x):
        return self.net(x)

model = CreditMLP(X_train.shape[1]).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

CreditMLP(
  (net): Sequential(
    (0): Linear(in_features=20, out_features=128, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): ReLU()
    (10): Linear(in_features=32, out_features=1, bias=True)
  )
)

Total parameters: 13,441


In [6]:
# Training
pos_weight = torch.tensor([(1 - y_train.mean()) / y_train.mean()]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
test_ds  = TensorDataset(torch.FloatTensor(X_test),  torch.FloatTensor(y_test))
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=256)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(100):
    model.train()
    losses, correct, total = [], 0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb).squeeze()
        loss = criterion(logits, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
        correct += ((logits > 0).float() == yb).sum().item()
        total += len(yb)
    history['train_loss'].append(np.mean(losses))
    history['train_acc'].append(correct / total)

    model.eval()
    losses, correct, total = [], 0, 0
    with torch.no_grad():
        for xb, yb in test_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb).squeeze()
            losses.append(criterion(logits, yb).item())
            correct += ((logits > 0).float() == yb).sum().item()
            total += len(yb)
    history['val_loss'].append(np.mean(losses))
    history['val_acc'].append(correct / total)
    scheduler.step(history['val_loss'][-1])

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss {history['train_loss'][-1]:.4f} "
              f"Acc {history['train_acc'][-1]:.3f} | Val Loss {history['val_loss'][-1]:.4f} "
              f"Acc {history['val_acc'][-1]:.3f}")

Epoch  20 | Train Loss 0.5413 Acc 0.796 | Val Loss 0.8837 Acc 0.700
Epoch  40 | Train Loss 0.4522 Acc 0.831 | Val Loss 0.9665 Acc 0.760
Epoch  60 | Train Loss 0.4400 Acc 0.854 | Val Loss 1.0138 Acc 0.765
Epoch  80 | Train Loss 0.3973 Acc 0.868 | Val Loss 1.0263 Acc 0.765
Epoch 100 | Train Loss 0.4433 Acc 0.853 | Val Loss 1.0405 Acc 0.765


In [7]:
# Plot Training Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], color=MAIN_BLUE, label='Train Loss', lw=2)
ax1.plot(history['val_loss'],   color=IDA_RED,   label='Val Loss',   lw=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss'); ax1.legend()

ax2.plot(history['train_acc'], color=MAIN_BLUE, label='Train Acc', lw=2)
ax2.plot(history['val_acc'],   color=IDA_RED,   label='Val Acc',   lw=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Accuracy'); ax2.legend()

plt.suptitle('Credit MLP Training History', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch01_credit_training')

Saved nn_ch01_credit_training.pdf and nn_ch01_credit_training.png


In [8]:
# ROC Curve
from sklearn.metrics import roc_curve, auc, classification_report

model.eval()
with torch.no_grad():
    y_score = torch.sigmoid(model(torch.FloatTensor(X_test).to(device))).cpu().numpy().ravel()
y_pred = (y_score > 0.5).astype(int)

fpr, tpr, _ = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, color=MAIN_BLUE, lw=2.5, label=f'MLP (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='grey', ls='--', lw=1, label='Random')
ax.fill_between(fpr, tpr, alpha=0.15, color=MAIN_BLUE)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve - Credit Scoring MLP', fontweight='bold')
ax.legend(loc='lower right', fontsize=12)
save_fig('nn_ch01_credit_roc')

print(classification_report(y_test, y_pred, target_names=['Good', 'Bad']))

Saved nn_ch01_credit_roc.pdf and nn_ch01_credit_roc.png
              precision    recall  f1-score   support

        Good       0.86      0.79      0.83       140
         Bad       0.59      0.70      0.64        60

    accuracy                           0.77       200
   macro avg       0.73      0.75      0.73       200
weighted avg       0.78      0.77      0.77       200



In [9]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=['Good Credit', 'Bad Credit'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title('Confusion Matrix - Credit Scoring MLP', fontweight='bold')
save_fig('nn_ch01_credit_confusion')

Saved nn_ch01_credit_confusion.pdf and nn_ch01_credit_confusion.png


In [11]:
# ============================================================
# Permutation Importance (robust fix â€” no sklearn tag issues)
# ============================================================

import numpy as np
import torch
import matplotlib.pyplot as plt

from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator

# ------------------------------------------------------------
# Minimal, robust wrapper (NO ClassifierMixin)
# ------------------------------------------------------------
class TorchWrapper(BaseEstimator):
    def __init__(self, model, device):
        self.model = model
        self.device = device

    def fit(self, X, y):
        return self  # no training

    def predict_proba(self, X):
        self.model.eval()
        with torch.no_grad():
            X_t = torch.as_tensor(X, dtype=torch.float32, device=self.device)
            logits = self.model(X_t)
            probs = torch.sigmoid(logits).cpu().numpy().ravel()
        return np.column_stack([1 - probs, probs])


# ------------------------------------------------------------
# Custom scoring (bypass sklearn classifier detection)
# ------------------------------------------------------------
from sklearn.metrics import roc_auc_score

def auc_scorer(estimator, X, y):
    probs = estimator.predict_proba(X)[:, 1]
    return roc_auc_score(y, probs)


# ------------------------------------------------------------
# Run permutation importance
# ------------------------------------------------------------
wrapper = TorchWrapper(model, device)
wrapper.fit(X_test, y_test)

perm = permutation_importance(
    wrapper,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring=auc_scorer,   # ðŸ”´ KEY FIX
    n_jobs=1
)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
top_k = 15
idx = np.argsort(perm.importances_mean)

values = perm.importances_mean[idx[-top_k:]]
labels = [feature_names[i] for i in idx[-top_k:]]

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 8))

ax.barh(range(len(values)), values,
        color=MAIN_BLUE, edgecolor='white')

ax.set_yticks(range(len(values)))
ax.set_yticklabels(labels)

ax.set_xlabel('Mean Permutation Importance (AUC decrease)')
ax.set_title('Top 15 Feature Importances - Credit MLP', fontweight='bold')

plt.tight_layout()

# ðŸ”´ SAVE ONLY (no show)
fig.savefig("credit_feature_importance.png", dpi=300, bbox_inches='tight')
fig.savefig("credit_feature_importance.pdf", dpi=300, bbox_inches='tight')

print("Saved: credit_feature_importance.png / .pdf")

Saved: credit_feature_importance.png / .pdf


## Summary

Trained an MLP on the **German Credit dataset** (1000 samples, 20 features) for credit scoring with class-weighted loss. Evaluated via ROC-AUC, confusion matrix, and permutation feature importance.